# اليوم الثاني: تحقّق بصدق قبل أن تحسّن النموذج
**SDA-DSC-211 · Tamweel Lite · Day 2 · 60 minutes**  
إعداد وتقديم **ميعاد المري | Meaad Al-Marri**

ستكشف خصائص لم تكن متاحة عند الطلب، وتفصل الماضي الناضج عن فترات التحقق اللاحقة، وتمنع ظهور العميل على جانبي كل طية. ثم تجري بحثًا محدودًا وتفسّر الأرقام التي تظهر لك؛ لا يوجد ترتيب مطلوب لنتائج النماذج.

**تسليمك:** تقرير التحقق، سجل البحث، الإعدادات المختارة وتبريرك المكتوب. افتح **جلسة CPU مجانية جديدة** واختر **Runtime → Run all**؛ لا تحتاج إلى تشغيل دفتر سابق أو ربط Drive. املأ إجاباتك بعد قراءة النتائج، ثم أعد خلية الحفظ.

| الوقت | مهمتك |
|---|---|
| 0–8 | جهّز البيئة واقرأ البيانات |
| 8–18 | راجع التسرب والتكرار |
| 18–30 | افحص تقسيم الزمن والعملاء ونضج النتائج |
| 30–45 | ابحث بحد8 تجارب و3 طيات و120 ثانية |
| 45–53 | قارن المتوسط والتشتت والتغطية |
| 53–60 | اكتب تفسيرك واحفظ الأدلة |

This lab evaluates future requests from customers excluded from each training fold. Search uses a separate historical customer cohort. Synthetic data only; no real lending decisions.

## 1. جهّز بيئتك | Setup
يتحقق الإعداد من إصدارات المكتبات وبصمات الملفات. إذا طُلبت إعادة الجلسة، اختر **Restart session** ثم شغّل من البداية. الخيار الافتراضيFAST: خيطاCPU وسقف200 شجرة. FULL اختياري بسقف600؛ يبقى البحث محدودًا بـ8 تجارب و120 ثانية في الوضعين. لا تحتاج إلىGPU أو أداة مدفوعة.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, os, runpy, sys, time, urllib.request, urllib.error

RUN_STARTED = time.perf_counter()
COURSE_REVISION = "fe0c0204e6076a7ac2139b7336485a097343fb8a"
MANIFEST_SHA256 = "86d15e813caf833b7d86512808176388d4123dc3c01c6525ee849a76e8322c59"
SETUP_SHA256 = "79608f8f9559d9e6eb09d51ee541ee12d7e9262ac57d1ca3a948c8ecb14f0f35"
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
ROOT = Path("/content/tamweel") if IN_COLAB else Path(os.environ.get("TAMWEEL_PROJECT_DIR", ".")).resolve()
setup_path = ROOT / "scripts/setup_colab.py"
setup_path.parent.mkdir(parents=True, exist_ok=True)
if not setup_path.exists() or hashlib.sha256(setup_path.read_bytes()).hexdigest() != SETUP_SHA256:
    setup_url = f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{COURSE_REVISION}/scripts/setup_colab.py"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(setup_url, timeout=45) as response:
                setup_bytes = response.read(200000)
            if hashlib.sha256(setup_bytes).hexdigest() != SETUP_SHA256:
                raise ValueError("Setup checksum mismatch. Reopen the released notebook.")
            setup_path.write_bytes(setup_bytes)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise RuntimeError("Download unavailable. Retry setup or upload the unchanged project files; see READINESS_GUIDE.")
            time.sleep(attempt + 1)
environment = runpy.run_path(str(setup_path))["prepare"](ROOT, COURSE_REVISION, MANIFEST_SHA256)
FAST_MODE, FULL_MODE, SEED, N_JOBS = True, False, 211, 2

from setup_colab import fetch_verified
LAB_REVISION = "1e1da4acbee8941bef07245b259fb6c27ced4ad7"
SUPPORT_HASHES = {'scripts/day2_validation.py': 'd2fd0353330c3629bdaed6c570528792955fce232540a85d33532a9432522f43', 'data/day2_optuna_example.csv': 'a59000eee4caced319735d237b8d121b2e0c7384acce93392d214ddbfe2f5b93', 'data/day2_optuna_example.json': 'a91a4d3a39ecd9cbab9bf902aaf5e2e060d4e21bcc93793057d5250aa002802c'}
for relative, sha256 in SUPPORT_HASHES.items():
    fetch_verified(
        f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{LAB_REVISION}/{relative}",
        ROOT / relative, sha256)
print("Day 2 setup verified | إعداد اليوم الثاني جاهز")

## 2. افحص زمن إتاحة الخاصية | Leakage audit
الهدف `default_within_90d` هو وقوع حدث تعثر **خلال90 يومًا بعد الطلب**، وليس الوصول إلى90 يوم تأخر. اقرأ `available_at` في القاموس: هل كنت تعرف الخاصية لحظة الطلب؟ الارتباط القوي وحده لا يثبت التسرب.

| نوع التسرب | ما تمنعه |
|---|---|
| الهدف | خاصية تصف النتيجة بعد حدوثها |
| المعالجة | تعلّم تعويض المفقود من صفوف التحقق |
| الزمن | التدريب على طلب أو نتيجة لم تكن متاحة بعد |
| العميل | وجود العميل في تدريب الطية والتحقق منها |
| التكرار | نسختان من الطلب نفسه على جانبي التقسيم |

نقرأ نسخة التدريب المتسربة لفهم المشكلة. لا نقرأ ملف التحدي أو نستخدمه في الاختيار اليوم. قد ينزله الإعداد للتحقق من سلامة حزمة البيانات فقط.

In [ ]:
import json, platform, zipfile
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from data_checks import validate_frame
from day2_validation import (ValidationConfig, BASE_PARAMS, leakage_audit,
    deduplicate_applications, make_time_group_split, reserve_tuning_pool,
    tune_on_reserved_pool, negative_controls, evaluate_outer, summarize_folds, reflection_check)

contract = json.loads((ROOT / "data/data_contract.json").read_text(encoding="utf-8"))
dictionary = pd.read_csv(ROOT / "data/feature_dictionary.csv")
dirty = pd.read_csv(ROOT / "data/tamweel_dirty.csv")
validate_frame(dirty, contract, "dirty")
audit = leakage_audit(dirty, dictionary)
assert not audit.action.eq("BLOCK_UNKNOWN").any(), "Review unknown fields before training."
leaked_features = audit.loc[audit.action.eq("DROP_POST_OUTCOME"), "feature"].tolist()
clean, duplicates = deduplicate_applications(dirty.drop(columns=leaked_features))
validate_frame(clean, contract, "train")
target = contract["target"]["name"]
FAST_MODE, FULL_MODE = True, False
assert FAST_MODE != FULL_MODE, "Choose exactly one mode."
config = ValidationConfig(max_trees=200 if FAST_MODE else 600, trials=8, timeout_seconds=120)
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
display(audit)
display(pd.DataFrame([{**duplicates, "predictors": int(audit.action.eq("KEEP").sum()),
                       "positive_rate": clean[target].mean()}]))
print("Explain which two fields you removed and when their information becomes available.")

## 3. جمّد أدوار البيانات | Time, group and maturity
لثلاث فترات تحقق متتابعة، درّب على الطلبات التي تحقق الشرط **تاريخ الطلب +90 يومًا < بداية التحقق**، ثم استبعد كل صف يخص عميلًا في تحقق تلك الطية. المساواة بالحد الزمني مستبعدة عمدًا.

هذا يقيس التعميم إلى عملاء لم يظهروا في تدريب الطية. قد يظهر عميل في أكثر من فترة تحقق؛ الضمان هو عدم اشتراكه بين تدريب وتحقق **الطية نفسها**. وهو تقييم محافظ قد يختلف عن خدمة عملاء عائدين.

تبقى الصفوف المبكرة فترة تمهيد. لا نملأها بتنبؤات تدريب ونسميهاOOF. حُذفت النسخ المتطابقة للحدث قبل التقسيم؛ النسخ المتعارضة لنفس المعرّف توقف التشغيل لتراجع مصدرها.

In [ ]:
folds = make_time_group_split(clean, horizon_days=contract["target"]["horizon_days"])
fold_audit = pd.DataFrame([fold["audit"] for fold in folds])
for fold in folds:
    tr, va = clean.loc[fold["train"]], clean.loc[fold["valid"]]
    assert not set(tr.customer_id) & set(va.customer_id)
    assert (pd.to_datetime(tr.application_date) + pd.Timedelta(days=90) <
            pd.Timestamp(fold["audit"]["validation_start"])).all()
fold_audit["validation_positive_rate"] = fold_audit.validation_positives / fold_audit.validation_rows
display(fold_audit)

plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.2})
fig, ax = plt.subplots(figsize=(9, 3.6), constrained_layout=True)
x = np.arange(len(fold_audit))
ax.bar(x - .18, fold_audit.train_rows, .36, label="Mature training, customers purged", color="#2667A0")
ax.bar(x + .18, fold_audit.validation_rows, .36, label="Later validation", color="#BA5A17")
ax.set(xticks=x, xticklabels=fold_audit.validation_start, ylabel="Requests", xlabel="Validation starts",
       title="Three forward folds · zero shared customers within each fold")
ax.set_ylim(0, fold_audit.train_rows.max() * 1.35)
ax.legend(fontsize=9)
fig.savefig(ARTIFACTS / "day2_fold_sizes.png", dpi=160)
plt.show()

## 4. احجز البحث قبل رؤية نتائج التحقق | Search cohort
نختار الإعدادات مرة واحدة من تاريخ سابق لأول تحقق، بعد اكتمال90 يومًا، ونستبعد **جميع عملاء فترات التحقق الثلاث** من مجموعة البحث. تستخدم المعرّفات لتحديد المجموعة فقط؛ لا تستخدم أهداف المستقبل لاختيار الإعدادات.

داخل مجموعة البحث ثلاث طيات زمنية تفصل العملاء. وداخل تدريب كل طية جزء آخر للإيقاف المبكر: يتعلم التعويض والأشجار من inner fit، ويختار العدد من inner stop علىlog loss، ثم يعيد التدريب على تدريب الطية كاملًا. لا تختار صفوف التحقق الخارجي المعالجة أو عدد الأشجار أو معاملات البحث.

هذا الحجز يترك مجموعة صغيرة؛ بعض الأجزاء الداخلية تحوي أمثلة موجبة قليلة. افحص الأعداد وسجّل هذا القيد. لا تساوِ أفضلAP في البحث بتقدير الأداء المستقبلي.

راجع [مخاطر تسرب المعالجة](https://scikit-learn.org/1.6/common_pitfalls.html) و[اختيار التحقق](https://scikit-learn.org/1.6/modules/cross_validation.html).

In [ ]:
pool = reserve_tuning_pool(clean, folds)
outer_ids = np.concatenate([fold["valid"] for fold in folds])
assert not set(pool.customer_id) & set(clean.loc[outer_ids, "customer_id"])
assert (pd.to_datetime(pool.application_date) + pd.Timedelta(days=90) < pd.Timestamp("2023-07-01")).all()
display(pd.DataFrame([{"search_rows": len(pool), "search_positives": int(pool[target].sum()),
    "search_positive_rate": pool[target].mean(), "latest_application": pool.application_date.max(),
    "outer_validation_rows": len(outer_ids)}]))

## 5. ابحث ضمن ميزانية واضحة | Optuna
نحسّن متوسط **Average Precision (AP)** عبر3 طيات للبحث. نغيّر `learning_rate` و`num_leaves` و`min_child_samples` فقط. البذرة211، والتجارب متتابعة، والحد8 تجارب أو120 ثانية للبحث. الميزانية لا تشمل التنزيل أو المقارنات التالية. الإيقاف الزمني تعاوني بين الجولات؛ قد يتجاوز الحد قليلًا بسبب كلفة إنهاء العملية.

`COMPLETE` تعني احتساب الطيات الثلاث. `PRUNED` تجربة أوقفت مبكرًا، و`FAIL` قد تنتج عن نفاد الميزانية أثناء التدريب؛ لا تعامل درجتها الجزئية كمتوسط مكتمل. نحفظ حالات كل المحاولات وأزمنتها.

إذا لم تكتمل أي تجربة، يعرض الدفتر مثالًا محفوظًا **للنقاش فقط**؛ لا ينسخه إلى نتائجك ولا يختار إعداداته لك. يظهر `TUNING_REQUIRED`، وتظل مقارنة الإعداد الثابت قابلة للحفظ. أعد لاحقًا من جلسةCPU جديدة؛ لا تشترِ اشتراكًا لحل التمرين. [توثيق حدود البحث](https://optuna.readthedocs.io/en/v4.5.0/reference/generated/optuna.study.Study.html).

In [ ]:
search, history = tune_on_reserved_pool(pool, contract, config)
has_tuned = search["best_params"] is not None
display(pd.DataFrame([{key: search[key] for key in ["status", "completed_trials", "attempted_trials",
    "elapsed_seconds", "stopping_reason", "best_tuning_ap"]}]))
display(pd.DataFrame(search["fold_audits"]))
display(history)
if not has_tuned:
    example = pd.read_csv(ROOT / "data/day2_optuna_example.csv")
    example_info = json.loads((ROOT / "data/day2_optuna_example.json").read_text(encoding="utf-8"))
    assert example_info["source"] == "EDUCATIONAL_EXAMPLE"
    print("EDUCATIONAL_EXAMPLE — discussion only; not your run or submission. TUNING_REQUIRED.")
    display(example)
else:
    print("Frozen search parameters:", search["best_params"])

fig, ax = plt.subplots(figsize=(9, 3.5), constrained_layout=True)
completed = history[history.state.eq("COMPLETE")]
if len(completed):
    ax.plot(completed.trial + 1, completed.mean_ap, marker="o", color="#2667A0", label="Completed trial mean AP")
    ax.step(completed.trial + 1, completed.mean_ap.cummax(), where="post", color="#BA5A17",
            linestyle="--", label="Best completed so far")
    ax.legend(fontsize=9)
else:
    ax.text(.5, .5, "No completed live trials — rerun needed", transform=ax.transAxes, ha="center")
ax.set(xlabel="Trial (1-based)", ylabel="Search AP", ylim=(0, 1), xticks=range(1, config.trials+1),
       title="Live search only · three internal folds · not outer performance")
fig.savefig(ARTIFACTS / "day2_search.png", dpi=160)
plt.show()

## 6. قارن بعد تثبيت البحث | Honest comparison
نستخدمLightGBM في كل الصفوف. أول تجربتين ضوابط تعليمية غير صالحة لاتخاذ قرار: العشوائية المتسربة تسمح بخاصيتين بعد النتيجة وتتعلم التعويض من كامل البيانات عمدًا؛ العشوائية النظيفة تصلح الخصائص والتعويض، لكنها تبقي اختلاط الزمن والعملاء. كلاهما بنفس الطيات العشوائية و80 شجرة.

المقارنة الزمنية الثابتة تستخدم الإعداد الأساسي نفسه و80 شجرة، ثم نضيف الإعدادات المجمدة من البحث مع اختيار الأشجار داخل التدريب. لا نفترض تحسن البحث أو انخفاض كل مقياس بعد التنظيف.

العشوائية تغطي10,000 طلب؛ الزمنية تغطي5,039 طلبًا لاحقًا، وأحجام تدريبها مختلفة. الفروق **وصفية** تتأثر بالفترة والسكان والبيانات المتاحة؛ ليست قياسًا سببيًا مستقلًا لأثر التسرب. لا تستخدم ضبطًا متسربًا لمجرد ارتفاع رقمه.

In [ ]:
controls = negative_controls(clean, dirty, contract, config)
fixed_report, fixed_oof, fixed_origin = evaluate_outer(
    clean, contract, folds, BASE_PARAMS, config, "clean_time_group_fixed", fixed_trees=80)
reports, predictions, origins = [controls, fixed_report], [fixed_oof], fixed_origin
if has_tuned:
    tuned_report, tuned_oof, tuned_origin = evaluate_outer(
        clean, contract, folds, search["best_params"], config, "clean_time_group_tuned")
    reports.append(tuned_report); predictions.append(tuned_oof); origins += tuned_origin
validation_report = pd.concat(reports, ignore_index=True)
summary = summarize_folds(validation_report)
oof = pd.concat(predictions, ignore_index=True)
assert not oof.duplicated(["scheme", "application_id"]).any()
display(summary.round(4))
display(validation_report[["scheme", "fold", "roc_auc", "average_precision", "train_rows",
    "validation_rows", "selected_trees", "shared_customers", "prediction_source"]].round(4))
gaps = summary.set_index("scheme")
print("Observed AP gap, leaky minus clean random:",
      round(gaps.loc["leaky_random_control", "ap_mean"] - gaps.loc["clean_random_control", "ap_mean"], 4))
print("Observed AP gap, clean random minus honest fixed:",
      round(gaps.loc["clean_random_control", "ap_mean"] - gaps.loc["clean_time_group_fixed", "ap_mean"], 4))

### اقرأ المتوسط والتشتت معًا
العلامة هي المتوسط غير الموزون للطيات الثلاث؛ الخط ± انحراف معياري عيّني `ddof=1`. الطيات مترابطة وليست ثلاث تجارب مستقلة، لذلك **الخط ليس فترة ثقة أو اختبار دلالة**. AP هنا هوAverage Precision، وليس مساحة شبه المنحرف. افحص حجم كل فترة ونسبة الموجب أيضًا.

قد يختلف عدد الأشجار أو آخر المنازل العشرية بين الأنظمة رغم تثبيت البذرة والإصدارات. لا تعِد البحث لأن رقم التحقق الخارجي لم يعجبك؛ عند تكرار الاختيار عليه يصبح جزءًا من التطوير، وليس اختبارًا نهائيًا.

In [ ]:
labels = {"leaky_random_control": "Leaky random (unsafe)", "clean_random_control": "Clean random (unsafe)",
          "clean_time_group_fixed": "Honest, fixed 80 trees", "clean_time_group_tuned": "Honest, reserved search"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
y = np.arange(len(summary))
for ax, mean_col, sd_col, title in zip(axes, ["roc_auc_mean", "ap_mean"], ["roc_auc_sd", "ap_sd"], ["ROC-AUC", "Average precision"]):
    ax.errorbar(summary[mean_col], y, xerr=summary[sd_col], fmt="o", capsize=5, color="#2667A0")
    ax.set(yticks=y, yticklabels=[labels[s] for s in summary.scheme], xlim=(0, 1.03),
           xlabel="Mean ± sample SD (not a confidence interval)", title=title)
    ax.invert_yaxis()
    for row, value in enumerate(summary[mean_col]):
        ax.annotate(f"{value:.3f}", (value, row), xytext=(-6, 10), textcoords="offset points", ha="right", fontsize=9)
fig.suptitle("Three folds per protocol · populations differ · descriptive comparison")
fig.savefig(ARTIFACTS / "day2_validation_comparison.png", dpi=160)
plt.show()

## 7. تحقّق من تغطية التنبؤات | OOF coverage
كل احتمال محفوظ حُسب لطلب خارج تدريب طيته وبعد تثبيت خياراتها الداخلية. لكل إعداد زمني5,039 احتمالًا؛ بقية4,961 طلبًا فترة تمهيد بلاOOF. لا تضع احتمالًا مصطنعًا أو تنبؤ تدريب مكانها. لا تُصدّر تنبؤات الضوابط المتسربة للاستخدام اللاحق.

OOF هنا تغطي50.39% من كل التدريب، و100% من الطلبات المؤهلة بدءًا من2023-07-01. ليست اختبارًا نهائيًا مستقلًا ولا ضمانًا لعدالة النموذج أو معايرته. احتفظ بدور كل صف لتبني قراراتك اللاحقة على التغطية المعلنة.

In [ ]:
coverage = clean[["application_id", "application_date"]].copy()
coverage["role"], coverage["fold"] = "warmup_no_oof", 0
for fold in folds:
    coverage.loc[fold["valid"], "role"] = "eligible_oof"
    coverage.loc[fold["valid"], "fold"] = fold["audit"]["fold"]
eligible_ids = set(coverage.loc[coverage.role.eq("eligible_oof"), "application_id"])
for scheme, group in oof.groupby("scheme"):
    assert set(group.application_id) == eligible_ids
    assert group.probability.between(0, 1).all()
display(pd.DataFrame([{"all_rows": len(clean), "eligible_rows": len(eligible_ids),
    "warmup_without_oof": len(clean)-len(eligible_ids), "whole_data_coverage": len(eligible_ids)/len(clean),
    "eligible_coverage": 1.0, "live_honest_schemes": oof.scheme.nunique()}]))

## 8. اكتب تفسيرك | Your reasoning
عدّل الخانات بكلماتك، ثم أعد هذه الخلية وخلية الحفظ:

1. سمِّ مثالًا للتسرب، واشرح لماذا لم تكن معلومته متاحة عند القرار.
2. برّر فصل الزمن والعملاء واشتراط نضج90 يومًا.
3. اذكر فجوتين بالأرقام من جدولك، وفسّر لماذا لا تضمنان تفوق إعداد على كل البيانات.
4. لماذا تتعلم المعالجة داخل التدريب؟ وما دور جزء الإيقاف الداخلي؟
5. اذكر قيدًا في حجم البحث وقيدًا في تغطيةOOF أو اعتماد الطيات.

التحقق التالي يفحص اكتمال الخانات فقط؛ لا يكتب إجاباتك ولا يمنح درجة.

In [ ]:
responses = {"leak_example": "", "split_reason": "", "observed_gap": "",
             "imputer_reason": "", "limitations": ""}
checkpoint = reflection_check(responses)
print(checkpoint["status"])
print("Missing:", checkpoint["missing"])

## 9. احفظ أدلتك | Export
نزّل `day2_artifacts.zip` من **Files → tamweel → artifacts**، وافكّه وارفع ملفاته إلى `artifacts/` في مستودعك. احفظ الدفتر من **File → Download → Download .ipynb** باسم `notebooks/02_validation_tuning.ipynb`.

أهم الملفات: `validation_report.csv` لكل طية، و`validation_summary.csv` للمتوسط والتشتت، و`best_params.json` و`optuna_results.csv` للبحث الفعلي. تضم الحزمة التدقيق والتغطية وتنبؤاتOOF ومصدر التدريب والرسوم وتفسيرك وبيئة التشغيل.

`TECHNICAL_READY` نجاح التشغيل والبحث؛ `TUNING_REQUIRED` يعني أن البحث لم ينتج تجربة مكتملة بعد. `LEARNER_WORK_REQUIRED` يعني بقاء إجاباتك؛ `READY_FOR_REVIEW` اكتمال الحقول فقط. راجع [دليل اليوم الثاني](https://github.com/almiyead-rgb/sda-dsc-211-student-template/blob/main/DAY2_GUIDE.md) للتسليم. احتفظ بنسخة قبل إعادة التجربة؛ الحفظ يستبدل ملفات هذا اليوم.

In [ ]:
checkpoint = reflection_check(responses)
technical_status = "TECHNICAL_READY" if has_tuned else "TUNING_REQUIRED"
csv_outputs = {"validation_report.csv": validation_report, "validation_summary.csv": summary,
    "optuna_results.csv": history, "leakage_audit.csv": audit, "fold_audit.csv": fold_audit,
    "day2_oof_predictions.csv": oof, "day2_oof_coverage.csv": coverage}
for filename, table in csv_outputs.items():
    table.to_csv(ARTIFACTS / filename, index=False)
run = {"technical_status": technical_status, "learner_status": checkpoint["status"],
    "data_revision": COURSE_REVISION, "lab_revision": LAB_REVISION, "support_sha256": SUPPORT_HASHES,
    "data_manifest_sha256": MANIFEST_SHA256,
    "dirty_data_sha256": hashlib.sha256((ROOT / "data/tamweel_dirty.csv").read_bytes()).hexdigest(),
    "config": asdict(config), "mode": "FAST" if FAST_MODE else "FULL", "python": platform.python_version(),
    "duplicates": duplicates, "removed_features": leaked_features,
    "oof_rows_per_scheme": {name: len(group) for name, group in oof.groupby("scheme")},
    "oof_whole_data_coverage": len(eligible_ids)/len(clean), "warmup_rows": len(clean)-len(eligible_ids),
    "search_source": "LIVE", "example_used_for_results": False,
    "challenge_used_for_modeling": False, "grading": "No automatic grade"}
json_outputs = {"best_params.json": search, "day2_provenance.json": origins,
                "day2_reflection.json": {"responses": responses, "checkpoint": checkpoint}, "day2_run.json": run}
for filename, payload in json_outputs.items():
    (ARTIFACTS / filename).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
files = list(csv_outputs) + list(json_outputs) + ["day2_fold_sizes.png", "day2_search.png",
    "day2_validation_comparison.png", "environment.json"]
with zipfile.ZipFile(ARTIFACTS / "day2_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in files:
        assert (ARTIFACTS / filename).is_file(), filename
        archive.write(ARTIFACTS / filename, arcname=filename)
print(technical_status, "|", checkpoint["status"])
print(f"Saved {len(files)} files in day2_artifacts.zip | احفظ الدفتر وملفاتك ثم أكمل تفسيرك")

## قبل أن تنتقل
اعرض مثال التسرب، ثم افتح تقرير الطيات وأشر إلى دليل فصل العملاء ونضج النتائج. وضّح الفرق بين عدد محاولات البحث وعدد التجارب المكتملة، وبين متوسطAP عبر الطيات وAP المحسوب بعد جمع احتمالاتها؛ لم نحسب الثاني هنا. ضع حدود النتيجة بجانبها.

**مراجع الأدوات:** [scikit-learn: التحقق](https://scikit-learn.org/1.6/modules/cross_validation.html) · [منع تسرب المعالجة](https://scikit-learn.org/1.6/common_pitfalls.html) · [LightGBM: الإيقاف المبكر](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.early_stopping.html) · [Optuna: البحث المحدود](https://optuna.readthedocs.io/en/v4.5.0/reference/generated/optuna.study.Study.html).

Your submission is your actual run plus your interpretation. A saved educational example is never a substitute for your own completed search.